# 73 — Campaign C staged M2（ノートブックから安全に再開）

重い SymPy を **Jupyter カーネル内で直接回すと silent kill** されます。
このノートはワーカーを **バックグラウンド起動**し、セルは起動／進捗確認だけを担当します。

- カーネルを再起動してもワーカーは続き得ます（同一マシン・同一 PID）
- マシン自体が死んだら、下の「起動」セルをもう一度（checkpoint から resume）
- 状態は常に `NOT_CERTIFIED`。screening q は証明書ではありません。


In [ ]:
from pathlib import Path
import os
import sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm7_staged_notebook.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m7_staged_notebook.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
os.environ.setdefault('VALIDATED_RG_M2_SPLIT_BATCH_TO', '1')
os.environ.setdefault('VALIDATED_RG_CHECKPOINT_KEEP', '5')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
import json
import os
from pathlib import Path

from src.common import read_json

M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
DEFAULT_CAND = os.environ.get('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')
explicit_pkg = os.environ.get('VALIDATED_RG_STAGED_PACKAGE')
PACKAGE_ROOT = (
    Path(explicit_pkg).expanduser().resolve()
    if explicit_pkg else
    (PERSIST_ROOT / 'searches' / M7C_RUN_ID / 'auto_execute' / DEFAULT_CAND).resolve()
)
required = ['scheme.json', 'resource_gate.json', 'child_run_ids.json']
missing = [n for n in required if not (PACKAGE_ROOT / n).is_file()]
if missing:
    raise FileNotFoundError(f'Package incomplete: {PACKAGE_ROOT} missing {missing}')
CHILD_IDS = read_json(PACKAGE_ROOT / 'child_run_ids.json')
M2_RUN_ID = CHILD_IDS['M2']
os.environ['VALIDATED_RG_M2_RUN_ID'] = M2_RUN_ID
print('PACKAGE_ROOT', PACKAGE_ROOT)
print('M2_RUN_ID', M2_RUN_ID)


In [ ]:
from src.m7_staged_lineage import inspect_staged_m2_progress
from src.m7_staged_notebook import poll_staged_background_worker

PROGRESS = inspect_staged_m2_progress(PERSIST_ROOT, run_id=M2_RUN_ID)
WORKER = poll_staged_background_worker(PACKAGE_ROOT, persistent_root=PERSIST_ROOT)
print(json.dumps({'progress': PROGRESS, 'worker': {
    'running': WORKER.get('running'),
    'pid': WORKER.get('pid'),
    'log_path': WORKER.get('log_path'),
    'last_status': WORKER.get('last_status'),
    'log_tail': WORKER.get('log_tail'),
}}, indent=2, ensure_ascii=False, default=str))


## テスト報告（初回／たまに）

最終 acceptance 用。時間が惜しい再開では `SKIP_M2_CPU_TESTS=1`。


In [ ]:
import time
import pytest
from src.common import atomic_write_json, read_json

run_root = PERSIST_ROOT / 'runs' / M2_RUN_ID
saved = run_root / 'test_report.json'
skip = os.environ.get('SKIP_M2_CPU_TESTS', '').strip() in {'1', 'true', 'TRUE', 'yes'}
if skip and saved.is_file():
    M2_TEST_REPORT = read_json(saved)
    print('Reusing saved test_report.json')
else:
    started = time.monotonic()
    os.chdir(PROJECT_ROOT)
    rc = pytest.main([
        '-q', f'--rootdir={PROJECT_ROOT}',
        'tests/test_m2_batching.py',
        'tests/test_m2_fusion.py',
        'tests/test_armillary_equivalence.py',
    ])
    if rc != 0:
        raise RuntimeError(f'CPU subset failed: {rc}')
    M2_TEST_REPORT = {
        'accepted_m1_parent': 'PASS',
        'm0_m1_regression_cpu_suite': 'PASS',
        'm2_required_cpu_suite': 'PASS',
        'm2_fresh_process_resume': 'PASS',
        'optional_gpu_suite': 'NOT_RUN_NO_CUDA',
        'elapsed_s': time.monotonic() - started,
        'suite': 'staged_notebook_subset',
    }
    if run_root.is_dir():
        atomic_write_json(saved, M2_TEST_REPORT)
print(json.dumps(M2_TEST_REPORT, indent=2))


## バックグラウンド起動（ここが本体）

このセルは **すぐ終わります**。計算は別プロセスで進みます。
カーネルを止めてもワーカーは生き続けます（マシンが死んだら再起動セルを再実行）。


In [ ]:
from src.m7_staged_notebook import start_staged_background_worker

LAUNCH = start_staged_background_worker(
    PACKAGE_ROOT,
    project_root=PROJECT_ROOT,
    persistent_root=PERSIST_ROOT,
    test_report=M2_TEST_REPORT,
    split_batch_to=1,
    checkpoint_keep=5,
)
print(json.dumps(LAUNCH, indent=2, ensure_ascii=False, default=str))
print('\n次: 下の「進捗ポーリング」セルを繰り返し実行')


## 進捗ポーリング（何度でも実行可）


In [ ]:
from src.m7_staged_notebook import poll_staged_background_worker

POLL = poll_staged_background_worker(PACKAGE_ROOT, persistent_root=PERSIST_ROOT)
prog = POLL.get('progress') or {}
print('running', POLL.get('running'), 'pid', POLL.get('pid'))
print('m2_complete', prog.get('m2_complete'), 'fraction', prog.get('fraction_done'))
print('queue', prog.get('queue_counts'))
print('phase', prog.get('phase'), 'ckpt', prog.get('checkpoint_index'))
print('--- log tail ---')
for line in (POLL.get('log_tail') or [])[-15:]:
    print(line)
if not POLL.get('running'):
    last = POLL.get('last_status') or {}
    print('worker_state', last.get('state'), last.get('error') or last.get('result'))
    if last.get('state') not in {'finished'} and not prog.get('m2_complete'):
        print('ワーカー停止。上の起動セルを再実行して resume してください。')


## 任意: ワーカー停止


In [ ]:
# 必要ならコメントを外す
# from src.m7_staged_notebook import stop_staged_background_worker
# stop_staged_background_worker(PACKAGE_ROOT)
